# A quick look at the photo features

These are simple generated images so it's easy to see what changes. Let's try blur, color, and a few grouping settings.

In [1]:
# Load the local tools
import sys
from pathlib import Path

root = Path.cwd()
if not (root / "features.py").exists():
    root = root.parent
sys.path.insert(0, str(root))
from features import scan_images
from group import group_images
records, errors = scan_images(root / "examples/demo")
print(f"{len(records)} images loaded")

7 images loaded


## Sharpness and blur

Laplacian variance responds to local edges. Blur should lower it, but texture and noise can increase it too. It is not an aesthetic score.

In [2]:
lookup = {r["file"]: r for r in records}
for name in ["scene.png", "scene_jpeg.jpg", "scene_soft.png"]:
    print(f"{name:20} {lookup[name]['sharpness']:.6f}")

scene.png            0.008502
scene_jpeg.jpg       0.008620
scene_soft.png       0.000017


## Why use color as well as a hash?

Uniform images contain no directional intensity changes, so very different colors can share the same difference hash.

In [3]:
# Check why the color comparison matters
from group import similar
red, blue = lookup["red.png"], lookup["blue.png"]
print("Same difference hash:", red["dhash"] == blue["dhash"])
print("Grouped as similar:", similar(red, blue))

Same difference hash: True
Grouped as similar: False


## Threshold sensitivity

Print group membership rather than just the number of groups. A smaller count is not automatically better.

In [4]:
# Try a few grouping thresholds
for threshold in [0.01, 0.04, 0.08, 0.15]:
    groups = group_images(records, max_pixel_rmse=threshold)
    print(f"RGB RMSE <= {threshold}: {len(groups)} groups")
    for group in groups:
        print("  " + ", ".join(records[i]["file"] for i in group))

RGB RMSE <= 0.01: 4 groups
  blue.png
  different_scene.png
  red.png
  scene.png, scene_copy.png, scene_jpeg.jpg, scene_soft.png
RGB RMSE <= 0.04: 4 groups
  blue.png
  different_scene.png
  red.png
  scene.png, scene_copy.png, scene_jpeg.jpg, scene_soft.png
RGB RMSE <= 0.08: 4 groups
  blue.png
  different_scene.png
  red.png
  scene.png, scene_copy.png, scene_jpeg.jpg, scene_soft.png
RGB RMSE <= 0.15: 4 groups
  blue.png
  different_scene.png
  red.png
  scene.png, scene_copy.png, scene_jpeg.jpg, scene_soft.png


## What I'd try next

Use photos from different shoots and label the pairs that really belong together. Then see whether embeddings help with the crops and lighting changes this approach struggles with.